# X Post Generator

In [1]:
from langgraph.graph import StateGraph,START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
import operator
import os

# Use the dedicated Groq integration package
from langchain_groq import ChatGroq

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## Get LLM 

In [3]:
def get_groq_llm():
    # ChatGroq automatically formats JSON/Tool schemas natively for Groq
    return ChatGroq(
        model="llama3-70b-8192", # Or your preferred Groq open-source engine
        api_key=os.getenv("GROQ_API_KEY"),
        max_tokens=1000,
        temperature=0.2,
        groq_api_base="https://api.groq.com/openai", # Explicitly anchors the correct base path
    )

llm = get_groq_llm()

## State (Shared Memory)

In [4]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: str
    feedback: str
    iteration: int
    max_iteration: int

    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]

## Structured Definition

In [5]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="Constructive feedback for the tweet.") # ✅ Fixed capitalization

In [6]:
def tweet_generator(state: TweetState):
    """Generates the initial draft text."""
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
            Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

            Rules:
            - Do NOT use question-answer format.
            - Max 280 characters.
            - Use observational humor, irony, sarcasm, or cultural references.
            - Think in meme logic, punchlines, or relatable takes.
            - Use simple, day to day english
            """)
    ]
    
    # Standard LLM call returns an AIMessage; fetch text content via .content
    response = llm.invoke(messages)
    raw_tweet = response.content

    return {
        'tweet': raw_tweet, 
        'tweet_history': [raw_tweet]
    }

In [7]:
def tweet_evaluator(state: TweetState):
    """Evaluates text quality and outputs structured JSON schemas."""
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""
            Evaluate the following tweet:
    
            Tweet: "{state['tweet']}"
    
            Use the criteria below to evaluate the tweet:
    
            1. Originality – Is this fresh, or have you seen it a hundred times before?
            2. Humor – Did it genuinely make you smile, laugh, or chuckle?
            3. Punchiness – Is it short, sharp, and scroll-stopping?
            4. Virality Potential – Would people retweet or share it?
            5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?
    
            Auto-reject if:
            - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
            - It exceeds 280 characters
            - It reads like a traditional setup-punchline joke
            - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)
    
            ### Respond ONLY in structured format:
            - evaluation: "approved" or "needs_improvement"
            - feedback: One paragraph explaining the strengths and weaknesses
            """)
    ]
    # This invocation returns a direct TweetEvaluation Pydantic object instance
    structured_response = structured_evaluator.invoke(messages)

    # ✅ Both attributes are now lowercase, matching the schema definitions perfectly
    return {
        'evaluation': structured_response.evaluation, 
        'feedback': structured_response.feedback, 
        'feedback_history': [structured_response.feedback]
    }

In [8]:
def tweet_evaluator(state: TweetState):
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""
            Evaluate the following tweet:
    
            Tweet: "{state['tweet']}"
    
            Use the criteria below to evaluate the tweet:
    
            1. Originality – Is this fresh, or have you seen it a hundred times before?
            2. Humor – Did it genuinely make you smile, laugh, or chuckle?
            3. Punchiness – Is it short, sharp, and scroll-stopping?
            4. Virality Potential – Would people retweet or share it?
            5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?
    
            Auto-reject if:
            - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
            - It exceeds 280 characters
            - It reads like a traditional setup-punchline joke
            - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)
    
            ### Respond ONLY in structured format:
            - evaluation: "approved" or "needs_improvement"
            - feedback: One paragraph explaining the strengths and weaknesses
            """)
    ]
    
    # Executes cleanly through Groq's schema validator endpoint
    structured_response = structured_evaluator.invoke(messages)

    return {
        'evaluation': structured_response.evaluation, 
        'feedback': structured_response.feedback, 
        'feedback_history': [structured_response.feedback]
    }

In [9]:
def tweet_evaluator(state: TweetState):
    """Evaluates text quality and outputs structured JSON schemas."""
    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""
            Evaluate the following tweet:
    
            Tweet: "{state['tweet']}"
    
            Use the criteria below to evaluate the tweet:
    
            1. Originality – Is this fresh, or have you seen it a hundred times before?
            2. Humor – Did it genuinely make you smile, laugh, or chuckle?
            3. Punchiness – Is it short, sharp, and scroll-stopping?
            4. Virality Potential – Would people retweet or share it?
            5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?
    
            Auto-reject if:
            - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
            - It exceeds 280 characters
            - It reads like a traditional setup-punchline joke
            - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)
    
            ### Respond ONLY in structured format:
            - evaluation: "approved" or "needs_improvement"
            - feedback: One paragraph explaining the strengths and weaknesses
            """)
    ]
    # This invocation returns a direct TweetEvaluation Pydantic object instance
    structured_response = structured_evaluator.invoke(messages)

    return {
        'evaluation': structured_response.evaluation, 
        'feedback': structured_response.feedback, 
        'feedback_history': [structured_response.feedback]
    }

In [10]:
def tweet_optimizer(state: TweetState):
    """Punches up text based on history and review details."""
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
            Improve the tweet based on this feedback:
            "{state['feedback']}"
    
            Topic: "{state['topic']}"
            Original Tweet:
            {state['tweet']}
    
            Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
        """)
    ]
    
    response = llm.invoke(messages)
    improved_tweet = response.content
    incremented_turn = state['iteration'] + 1

    return {
        'tweet': improved_tweet, 
        'iteration': incremented_turn, 
        'tweet_history': [improved_tweet]
    }

In [11]:
def tweet_router(state: TweetState) -> Literal['approved', 'needs_improvement']:
    """Evaluates metrics to break or sustain cyclic loops."""
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    return 'needs_improvement'

## Graph

In [12]:
graph = StateGraph(TweetState)

# Register workflow stations
graph.add_node('generate', tweet_generator)
graph.add_node('evaluate', tweet_evaluator)
graph.add_node('optimize', tweet_optimizer)

# Configure edge connections
graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

# Use Path Map Dictionaries to name edge tracks for cleaner visualization outputs
graph.add_conditional_edges(
    'evaluate', 
    tweet_router, 
    {
        'approved': END, 
        'needs_improvement': 'optimize'
    }
)
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

In [13]:
initial_state = {
    "topic": "developer drinking coffee at midnight",
    "tweet": "",
    "evaluation": "",
    "feedback": "",
    "iteration": 1,
    "max_iteration": 3,
    "tweet_history": [],
    "feedback_history": []
}

In [14]:
final_output = workflow.invoke(initial_state)
print(f"Final Tweet Status: {final_output['evaluation'].upper()}")
print(f"Total Iterations: {final_output['iteration']}")
print(f"Final Tweet: {final_output['tweet']}")

NotFoundError: Error code: 404 - {'error': {'message': 'Unknown request URL: POST /openai/openai/v1/chat/completions. Please check the URL for typos, or see the docs at https://console.groq.com/docs/', 'type': 'invalid_request_error', 'code': 'unknown_url'}}

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Ensure a persistent memory checkpointer is initialized during compilation
# memory_storage = MemorySaver()
# workflow = graph.compile(checkpointer=memory_storage)

# Define a unique session thread
thread_config = {"configurable": {"thread_id": "creative_session_101"}}

# 1. Initialize the graph state with your core variables
initial_state = {
    "topic": "A developer drinking coffee at midnight",
    "tweet": "",
    "evaluation": "",
    "feedback": "",
    "iteration": 1,
    "max_iteration": 3,
    "tweet_history": [],
    "feedback_history": []
}

print("--- STARTING WORKFLOW INVOCATION ---")
# Invoke the workflow dynamically using the thread configuration
final_state = workflow.invoke(initial_state, config=thread_config)

print(f"\nExecution Concluded.")
print(f"Final Loop Iteration Count: {final_state['iteration']}")
print(f"Final Evaluator Decision: {final_state['evaluation'].upper()}")
print(f"Final Output: {final_state['tweet']}")

--- STARTING WORKFLOW INVOCATION ---

Execution Concluded.
Final Loop Iteration Count: 2
Final Evaluator Decision: APPROVED
Final Output: Midnight IDE: “Stack overflow!”  
My mug: “Hold my espresso.” ☕️💥  
When the only thing running faster than my code is my caffeine‑induced panic mode. #devlife #nightshift


In [ ]:
thread_config_failsafe = {"configurable": {"thread_id": "token_guard_session_99"}}

# Artificially jumpstart the graph at the maximum iteration limit
failsafe_state = {
    "topic": "Quantum physics explained in a modern meme format",
    "tweet": "Standard unstructured text draft that will intentionally fail validation...",
    "evaluation": "needs_improvement",
    "feedback": "The text reads like a traditional joke setup. Completely rewrite.",
    "iteration": 3,  # ⚠️ Equals max_iteration boundary
    "max_iteration": 3,
    "tweet_history": ["Draft 1", "Draft 2"],
    "feedback_history": ["Feedback 1", "Feedback 2"]
}

print("--- TESTING FAILSAFE GUARDRAIL ---")
# Invoke the graph directly at the evaluation node stage
result_state = workflow.invoke(failsafe_state, config=thread_config_failsafe)

print(f"\nFailsafe Test Complete.")
print(f"Evaluator Score: {result_state['evaluation'].upper()}")
print(f"Total Completed Iterations: {result_state['iteration']}")
print(f"Final Action: Pushed to termination node to preserve token limits.")

--- TESTING FAILSAFE GUARDRAIL ---


BadRequestError: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}